# 02 · GLiNER baselines
Each GLiNER family runs in its own venv (`scripts/make_env.sh`) as a subprocess that writes a sharded, resumable prediction cache. First a 5-doc smoke run per system, then full calibration and test runs, then scoring in the main environment (append-only ledger on Drive).

After the first successful `make_env`, commit `envs/<name>.lock` and the resolved model revisions (printed in each prediction meta) into `s1pii/configs/baselines.yaml`.

In [ ]:
# Shared session setup (run in every notebook)
import os, subprocess, pathlib
from google.colab import drive
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/s1pii'); DRIVE.mkdir(parents=True, exist_ok=True)
os.environ['S1PII_DATA'] = '/content/s1pii_data'        # local disk; synced to/from Drive
os.environ['S1PII_RESULTS'] = str(DRIVE / 'results')     # ledger + prediction cache on Drive
REPO = pathlib.Path('/content/s1-pii')
if not REPO.exists():
    from google.colab import userdata                    # add GH_TOKEN (fine-grained, read-only) in Colab secrets
    tok = userdata.get('GH_TOKEN')
    subprocess.run(['git', 'clone', f'https://{tok}@github.com/anitksahu/s1-pii.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', 'https://github.com/anitksahu/s1-pii.git'], check=True)
os.chdir(REPO)
subprocess.run(['rsync', '-a', f'{DRIVE}/data/', os.environ['S1PII_DATA'] + '/'], check=False)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


In [ ]:
!pip install -q -e '.[data]'
!bash scripts/make_env.sh gliner && bash scripts/make_env.sh gliner2

In [ ]:
import json, subprocess, os
from s1pii import bench
from s1pii.adapters.base import load_config
SYSTEMS = {s: c['env'] for s, c in load_config()['systems'].items()}
DATASETS = ['tab_direct', 'spy_medical', 'spy_legal', 'pii_trace', 'nemotron']
OUT = os.environ['S1PII_RESULTS'] + '/predictions'
def run(system, docs, limit=None):
    cmd = [f'envs/{SYSTEMS[system]}/bin/python', '-m', 's1pii.adapters.run', '--system', system, '--docs', docs, '--out', OUT]
    if limit: cmd += ['--limit', str(limit), '--out', '/content/smoke']
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode: print(r.stdout[-2000:], r.stderr[-4000:]); raise RuntimeError(system)
    return json.loads(r.stdout.strip().splitlines()[-1])['predictions']

In [ ]:
# smoke: 5 docs per system on TAB test
for s in SYSTEMS:
    p = run(s, bench.split_paths('tab_direct')['test'], limit=5)
    from s1pii.ledger import read_predictions
    meta, preds = read_predictions(__import__('pathlib').Path(p))
    print(s, meta['revision'], meta['versions'], meta['report'])

In [ ]:
# full runs (resumable: re-run this cell after a disconnect)
PRED = {}
for s in SYSTEMS:
    for d in DATASETS:
        paths = bench.split_paths(d)
        PRED[(s, d)] = {part: run(s, str(paths[part])) for part in ('calib', 'test')}
        print(s, d, 'done')

In [ ]:
import pandas as pd, pathlib
rows = []
for (s, d), p in PRED.items():
    if s == 'nvidia_gliner_pii' and d == 'nemotron':
        continue   # trained on Nemotron-PII: excluded (prereg)
    r = bench.score(s, d, pathlib.Path(p['calib']), pathlib.Path(p['test']))['result']
    op = r['operating_points']['default']
    rows.append({'system': s, 'dataset': d, 'pAUC': r['pauc'], 'pAUC_ci': r['pauc_ci'], 'reach_over': r['reachable_max_over'],
                 'leak@default': op['leak'], 'over@default': op['over'], 'span_exposure': op['span_exposure']})
pd.DataFrame(rows).sort_values(['dataset', 'pAUC'])